In [2]:
predictions = [
    {"id": "A", "box": [0, 0, 100, 100],     "score": 0.95, "class_id": 0},
    {"id": "B", "box": [5, 5, 105, 105],     "score": 0.90, "class_id": 0},
    {"id": "C", "box": [200, 0, 300, 100],   "score": 0.80, "class_id": 0},
    {"id": "D", "box": [0, 0, 100, 100],     "score": 0.75, "class_id": 1},
    {"id": "E", "box": [400, 0, 500, 100],   "score": 0.20, "class_id": 0},
    {"id": "F", "box": [600, 0, 700, 100],   "score": 0.60, "class_id": 0},
]

CONF_THRESHOLD = 0.25

filtered = [
    p for p in predictions
    if p["score"] >= CONF_THRESHOLD
]

print("After confidence filter:", [p["id"] for p in filtered])

After confidence filter: ['A', 'B', 'C', 'D', 'F']


In [3]:
def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    
    inter_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0, min(ay2, by2) - max(ay1, by1))
    intersection = inter_w * inter_h

    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0

print(round(iou_xyxy(
    predictions[0]["box"],
    predictions[1]["box"]
), 4))

0.8223


**Implement NMS manually**

In [4]:
def manual_nms(predictions, iou_threshold=0.5, class_aware=True):
    remaining = sorted(
        predictions,
        key=lambda p: p["score"],
        reverse=True
    )
    kept = []
    
    while remaining:
        best = remaining.pop(0)
        kept.append(best)
        
        survivors = []
        for candidate in remaining:
            compare = (
                not class_aware
                or candidate["class_id"] == best["class_id"]
            )
            
            overlap = iou_xyxy(best["box"], candidate["box"])
            if not (compare and overlap > iou_threshold):
                survivors.append(candidate)
        
        remaining = survivors
    
    return kept

In [5]:
kept = manual_nms(filtered, iou_threshold=0.5)
agnostic_kept = manual_nms(
    filtered,
    iou_threshold=0.5,
    class_aware=False
)

print("Class-aware   :", [p["id"] for p in kept])
print("Class-agnostic:", [p["id"] for p in agnostic_kept])

Class-aware   : ['A', 'C', 'D', 'F']
Class-agnostic: ['A', 'C', 'F']


In [6]:
for threshold in [0.3, 0.5, 0.9]:
    result = manual_nms(
        filtered,
        iou_threshold=threshold
    )
    print(threshold, [p["id"] for p in result])

0.3 ['A', 'C', 'D', 'F']
0.5 ['A', 'C', 'D', 'F']
0.9 ['A', 'B', 'C', 'D', 'F']


**Before/after visualization**